In [ ]:
import os
from pathlib import Path
import amulet
from tqdm import tqdm
import random
from collections import defaultdict, Counter

random.seed(1984)



blocks_counter = Counter()

directory = Path("./samples")


files = os.listdir(directory)
random.shuffle(files)

limit = len(files)

for i, file in enumerate(tqdm(files)):
    if i >= limit:
        break
    path = directory / file
    try:
        construction = amulet.load_level(str(path))
    except Exception as e:
        print()
        print(path)
        print(e)

    dimension = construction.dimensions[0]
    selection = construction.bounds(dimension)
    for coords in selection.blocks:
        block = construction.get_block(*coords, dimension)
        blocks_counter[block.base_name] += 1




In [3]:
print(f"Total blocks: {len(blocks_counter)}")
print(f"Total amount: {blocks_counter.total()}")
print(f"20 Most common:")
print(*blocks_counter.most_common(20), sep="\n")

Total blocks: 181
Total amount: 785547479
20 Most common:
('air', 448983638)
('stone', 129865458)
('water', 95308840)
('dirt', 41823976)
('leaves', 13181000)
('grass_block', 12622500)
('sand', 6142543)
('sandstone', 4509040)
('gravel', 3154077)
('stained_terracotta', 3107841)
('andesite', 2921855)
('diorite', 2817411)
('granite', 2743257)
('terracotta', 2039881)
('coal_ore', 1867598)
('plant', 1827383)
('packed_ice', 1683301)
('log', 1680110)
('snow', 1428070)
('snow_block', 1224723)


In [4]:
minus_counter = Counter({key: -value for key, value in blocks_counter.items()})

minus_counter.most_common(20)

[('fletching_table', -1),
 ('lectern', -1),
 ('sapling', -1),
 ('stained_glass', -1),
 ('sculk_sensor', -1),
 ('sculk_shrieker', -1),
 ('small_amethyst_bud', -2),
 ('large_amethyst_bud', -3),
 ('barrel', -3),
 ('jack_o_lantern', -4),
 ('smithing_table', -4),
 ('smoker', -5),
 ('repeater', -5),
 ('amethyst_cluster', -5),
 ('redstone_torch', -6),
 ('wall_sign', -6),
 ('dispenser', -7),
 ('blast_furnace', -8),
 ('bookshelf', -8),
 ('medium_amethyst_bud', -8)]

Разобраться с дефольными атрибутами и зафиксировать финальный набор

Глянуть распределение атрибутов в мирах

Несколько вариантов кодирования блоков + атрибутов

СОхранить json всех блоков

# Раскинуть файлы по папкам

In [2]:
import os
from pathlib import Path
from tqdm import tqdm
import os

data_dir = Path("../../data/web_data/")


extensions = [".schematic", ".schem", ".litematic"]

In [ ]:
for ex in extensions:
    os.makedirs(data_dir / ex[1:], exist_ok=True)


for file in os.listdir(data_dir):
    file_path = data_dir / file
    if file_path.suffix in extensions:
        os.rename(file_path, data_dir / file_path.suffix[1:] / file)

## запустил скрипты


In [ ]:
for file in tqdm(os.listdir(data_dir / "litematic")):
    file_path = data_dir / "litematic" / file
    if file_path.suffix == ".schem":
        os.rename(file_path, data_dir / "schem" / file)

100%|██████████| 4064/4064 [00:08<00:00, 463.05it/s]


In [11]:
count = 0
for file in tqdm(os.listdir(data_dir / "schematic")):
    file_path = data_dir / "schematic" / file
    if file_path.suffix == ".schem":
        count += 1
        os.rename(file_path, data_dir / "schem" / file)
print(count)

100%|██████████| 11215/11215 [00:00<00:00, 144811.05it/s]

4


In [8]:
import nbtlib


schem = nbtlib.load(data_dir / "schem_py_2" / "100.schem")

In [17]:
schem["Schematic"]["Length"]

Short(30)

In [18]:
schem = nbtlib.load(data_dir / "schem" / "100.schem")

In [19]:
schem.keys()

dict_keys(['Version', 'DataVersion', 'Width', 'Height', 'Length', 'Palette', 'BlockData'])

## считаем статистики по schem_all

In [1]:
import os
from pathlib import Path
import amulet
import json
from tqdm import tqdm
import random
from collections import defaultdict, Counter
from logging import Logger
import numpy as np

random.seed(1984)
np.random.seed(1984)

with open("../block_data/filtered_blocks.json") as f:
    filtered_blocks = json.load(f)


blocks_counter = Counter()

directory = Path("../../data/web_data/schem_all")

shapes = []

files = os.listdir(directory)
np.random.shuffle(files)

limit =  1000 # len(files)

count = 0
for i, file in enumerate(tqdm(files, total=limit)):
    
    if i >= limit:
        break
    path = directory / file
    try:
        construction = amulet.load_level(str(path))
        dimension = construction.dimensions[0]
        selection = construction.bounds(dimension)
        bounds_array = selection.bounds_array
        shape = np.abs(bounds_array[0] - bounds_array[1])

        shapes.append(shape[None, :])

        for coords in selection.blocks:
            block = construction.get_block(*coords, dimension).base_name
            if f"minecraft:{block}" in filtered_blocks:
                blocks_counter[block] += 1
            else:
                blocks_counter["air"] += 1
        count += 1
        construction.close()
    except Exception as e:
        print(f"Error: {e}\nFile: {file}")
        continue

shapes = np.concatenate(shapes, axis=0)

INFO - PyMCTranslate Version 377
  0%|          | 1/1000 [00:06<1:46:29,  6.40s/it]INFO - Loading level ../../data/web_data/schem_all/28223.schem
WARNING - Could not find translation information for block easyanvils:minecraft/anvil[facing="north"] to universal in PyMCTranslate.Version(java, (1, 21, 2)). If this is not a vanilla block ignore this message
WARNING - Could not find translation information for block easyanvils:minecraft/anvil[facing="south"] to universal in PyMCTranslate.Version(java, (1, 21, 2)). If this is not a vanilla block ignore this message
WARNING - Could not find translation information for block diagonalfences:minecraft/spruce_fence[east="true",north="false",north_east="false",north_west="false",south="false",south_east="false",south_west="false",waterlogged="false",west="false"] to universal in PyMCTranslate.Version(java, (1, 21, 2)). If this is not a vanilla block ignore this message
  0%|          | 2/1000 [00:06<45:20,  2.73s/it]  INFO - Loading level ../../da

Error: Could not find a matching format for ../../data/web_data/schem_all/18540.schem
File: 18540.schem


WARNING - Could not find translation information for block simplylight:illuminant_panel[facing="down",waterlogged="false"] to universal in PyMCTranslate.Version(java, (1, 14, 4)). If this is not a vanilla block ignore this message
WARNING - Could not find translation information for block lightestlamp:light_air to universal in PyMCTranslate.Version(java, (1, 14, 4)). If this is not a vanilla block ignore this message
WARNING - Could not find translation information for block lightestlamp:zeta_lamp to universal in PyMCTranslate.Version(java, (1, 14, 4)). If this is not a vanilla block ignore this message
WARNING - Could not find translation information for block lightestlamp:omega_lamp to universal in PyMCTranslate.Version(java, (1, 14, 4)). If this is not a vanilla block ignore this message
WARNING - Could not find translation information for block simplylight:illuminant_panel[facing="south",waterlogged="false"] to universal in PyMCTranslate.Version(java, (1, 14, 4)). If this is not a 

Error: Could not find a matching format for ../../data/web_data/schem_all/26640.schem
File: 26640.schem
Error: Could not find a matching format for ../../data/web_data/schem_all/23286.schem
File: 23286.schem


  2%|▎         | 25/1000 [02:26<2:27:38,  9.09s/it]INFO - Loading level ../../data/web_data/schem_all/21518.schem
INFO - Loading level ../../data/web_data/schem_all/19286.schem
INFO - Loading level ../../data/web_data/schem_all/16149.schem
  3%|▎         | 28/1000 [02:26<1:20:11,  4.95s/it]INFO - Loading level ../../data/web_data/schem_all/17143.schem


...

 98%|█████████▊| 983/1000 [36:40<00:21,  1.24s/it]INFO - Loading level ../../data/web_data/schem_all/19389.schem
INFO - Loading level ../../data/web_data/schem_all/16455.schem
 99%|█████████▉| 988/1000 [36:42<00:06,  1.81it/s]INFO - Loading level ../../data/web_data/schem_all/28430.schem
INFO - Loading level ../../data/web_data/schem_all/23853.schem


Error: Could not find a matching format for ../../data/web_data/schem_all/28430.schem
File: 28430.schem


 99%|█████████▉| 992/1000 [36:44<00:04,  1.79it/s]INFO - Loading level ../../data/web_data/schem_all/17606.schem
INFO - Loading level ../../data/web_data/schem_all/17592.schem
100%|█████████▉| 997/1000 [36:45<00:00,  3.78it/s]INFO - Loading level ../../data/web_data/schem_all/24755.schem
INFO - Loading level ../../data/web_data/schem_all/18312.schem


Error: Could not find a matching format for ../../data/web_data/schem_all/24755.schem
File: 24755.schem


100%|██████████| 1000/1000 [36:46<00:00,  2.21s/it]


In [2]:
count

952

In [3]:
print(f"Min x,y,z: {shapes.min(axis = 0)}")
print(f"Max x,y,z: {shapes.max(axis = 0)}")
print(f"Mean x,y,z: {shapes.mean(axis = 0)}")
print(f"Median x,y,z: {np.median(shapes,axis=0)}")

Min x,y,z: [1 1 1]
Max x,y,z: [429 384 674]
Mean x,y,z: [46.92864638 42.75236097 46.38929696]
Median x,y,z: [30. 24. 29.]


In [4]:
volumes: np.ndarray = np.prod(shapes, axis=1)

print(f"Min Volume: {volumes.min()}")
print(f"Max Volume: {volumes.max()}")
print(f"Mean Volume: {volumes.mean()}")
print(f"Median Volume: {np.median(volumes)}")

Min Volume: 1
Max Volume: 44701888
Mean Volume: 590478.2990556138
Median Volume: 19488.0


In [7]:
print(f"Uniqie blocks: {len(blocks_counter)}")
print(f"Total amount: {blocks_counter.total()}")
print(f"20 Most common:")
print(*blocks_counter.most_common(20), sep="\n")

Uniqie blocks: 297
Total amount: 562725119
20 Most common:
('air', 528487561)
('stone', 13883599)
('dirt', 3467187)
('water', 2573585)
('stone_bricks', 1470147)
('andesite', 1174747)
('coarse_dirt', 984999)
('grass_block', 960862)
('diorite', 859504)
('cobblestone', 741172)
('granite', 650026)
('smooth_stone', 525266)
('sand', 521124)
('smooth_quartz', 498875)
('sandstone', 440663)
('snow_block', 367333)
('gravel', 316601)
('glowstone', 264948)
('cobbled_deepslate', 202750)
('quartz_block', 193212)


In [6]:
print(f"20 Least common:")

minus_counter = Counter({key: -value for key, value in blocks_counter.items()})

minus_counter.most_common(20)

20 Least common:


[('sniffer_egg', -1),
 ('candle_cake', -3),
 ('sculk_shrieker', -5),
 ('calibrated_sculk_sensor', -5),
 ('sculk_catalyst', -9),
 ('sculk_sensor', -9),
 ('smooth_quartz_slab', -9),
 ('weathered_cut_copper', -12),
 ('conduit', -13),
 ('smooth_quartz_stairs', -18),
 ('respawn_anchor', -19),
 ('ancient_debris', -24),
 ('crimson_fungus', -33),
 ('nether_gold_ore', -33),
 ('exposed_cut_copper', -33),
 ('exposed_copper', -34),
 ('pitcher_crop', -38),
 ('turtle_egg', -39),
 ('powder_snow', -52),
 ('cake', -57)]

In [11]:
# from pathlib import Path

# # litematic = os.listdir("../../data/web_data/litematic")

# schematics_ids = list(map(lambda x: Path(x).stem,os.listdir("../../data/web_data/schematic")))

# for file in tqdm(os.listdir("../../data/web_data/schem_all")):
#     file = Path(file)
#     if file.stem in schematics_ids:
#         os.rename("../../data/web_data/schem_all/" / file, "../../data/web_data/schem_done/" / file)
#     # print(file)

100%|██████████| 12815/12815 [00:25<00:00, 503.62it/s]


Глянуть пересечение по блокам между множествами

Сделать фильтр на число заменямых блкоов на воздух (10% грань)

Пофильтровать по минимальному объему

Глянукть сколько сэмплов умешается в 64х64х64

